In [ ]:
import math

def ler_float(prompt):
    """Lê um número do usuário aceitando vírgula ou ponto como separador decimal."""
    return float(input(prompt).strip().replace(',', '.'))

def calcular_volume_tubulacao(diametro_externo, diametro_interno, comprimento):
    """
    Calcula o volume de material de uma tubulação (volume do metal).
    
    Parâmetros:
    diametro_externo (float): Diâmetro externo em metros.
    diametro_interno (float): Diâmetro interno em metros.
    comprimento (float): Comprimento da tubulação em metros.
    
    Retorna:
    float: Volume em metros cúbicos.
    """
    # Calcula a área da seção transversal do metal
    area_secao = (math.pi / 4) * (diametro_externo**2 - diametro_interno**2)

    # Calcula o volume
    return area_secao * comprimento

def calcular_volume_tubulacao_sch(diametro_nominal, schedule, comprimento, unidades='mm'):
    """
    Calcula o volume de material baseado no diâmetro nominal e schedule.
    Requer tabela de dimensões de tubos.
    """
    # Tabela simplificada de dimensões (diâmetro externo e espessura de parede em mm)
    # Fonte: ASTM A53/A106
    tabela_tubos = {
        ('1/2', '40'): {'de': 21.3, 'e': 2.77},
        ('3/4', '40'): {'de': 26.7, 'e': 2.87},
        ('1', '40'): {'de': 33.4, 'e': 3.38},
        ('1½', '40'): {'de': 48.3, 'e': 3.68},
        ('2', '40'): {'de': 60.3, 'e': 3.91},
        ('3', '40'): {'de': 88.9, 'e': 5.49},
        ('4', '40'): {'de': 114.3, 'e': 6.02},
        ('6', '40'): {'de': 168.3, 'e': 7.11},
        ('8', '40'): {'de': 219.1, 'e': 8.18},
        ('10', '40'): {'de': 273.0, 'e': 9.27},
        ('12', '40'): {'de': 323.8, 'e': 9.53},
        ('2', '80'): {'de': 60.3, 'e': 5.54},
        ('4', '80'): {'de': 114.3, 'e': 8.56},
        # Adicione mais tamanhos conforme necessário
    }
    
    try:
        chave = (str(diametro_nominal), str(schedule))
        if chave not in tabela_tubos:
            print("Tamanho/schedule não encontrado na tabela.")
            return None
        
        dados = tabela_tubos[chave]
        de = dados['de']
        e = dados['e']
        di = de - (2 * e)
        
        if unidades == 'mm':
            # Converter mm para metros
            de_m = de / 1000
            di_m = di / 1000
        else:
            de_m = de
            di_m = di
        
        volume = calcular_volume_tubulacao(de_m, di_m, comprimento)
        return volume, de, di, e
        
    except KeyError:
        print("Combinação diâmetro/schedule não suportada.")
        return None

def main():
    print("CALCULADORA DE VOLUME DE TUBULAÇÃO")
    print("=" * 40)
    
    while True:
        print("\nOpções:")
        print("1 - Calcular com dimensões conhecidas")
        print("2 - Calcular usando Schedule (Tabela)")
        print("3 - Sair")
        
        opcao = input("Escolha uma opção (1-3): ")
        
        if opcao == '1':
            try:
                de = ler_float("Diâmetro externo (mm): ") / 1000
                di = ler_float("Diâmetro interno (mm): ") / 1000
                comprimento = ler_float("Comprimento (m): ")
                
                volume = calcular_volume_tubulacao(de, di, comprimento)
                
                if volume is not None:
                    print(f"\nResultado:")
                    print(f"Volume de material: {volume:.6f} m³")
                    print(f"Volume de material: {volume * 1000:.3f} litros")
                    
            except ValueError:
                print("Erro: Insira valores numéricos válidos.")
                
        elif opcao == '2':
            try:
                diametro_nominal = input("Diâmetro nominal (ex: 2, 4, 6): ")
                schedule = input("Schedule (ex: 40, 80): ")
                comprimento = ler_float("Comprimento (m): ")
                
                resultado = calcular_volume_tubulacao_sch(diametro_nominal, schedule, comprimento)
                
                if resultado is not None:
                    volume, de, di, e = resultado
                    print(f"\nDados do tubo {diametro_nominal}\" SCH {schedule}:")
                    print(f"Diâmetro externo: {de} mm")
                    print(f"Diâmetro interno: {di:.2f} mm")
                    print(f"Espessura: {e} mm")
                    print(f"Volume de material: {volume:.6f} m³")
                    print(f"Volume de material: {volume * 1000:.3f} litros")
                    
            except ValueError:
                print("Erro: Insira valores válidos.")
                
        elif opcao == '3':
            print("Saindo...")
            break
            
        else:
            print("Opção inválida. Tente novamente.")

# Função adicional para cálculo em lote
def calcular_lote_tubulacoes(lista_tubos):
    """
    Calcula volume para múltiplas tubulações.
    
    Parâmetros:
    lista_tubos: Lista de tuplas (diâmetro_nominal, schedule, comprimento)
    """
    resultados = []
    total_volume = 0
    
    for tubo in lista_tubos:
        dn, sch, comp = tubo
        resultado = calcular_volume_tubulacao_sch(dn, sch, comp)
        
        if resultado is not None:
            volume, de, di, e = resultado
            resultados.append({
                'tubo': f"{dn}\" SCH {sch}",
                'comprimento': comp,
                'volume': volume
            })
            total_volume += volume
    
    return resultados, total_volume

if __name__ == "__main__":
    # Exemplo de uso direto
    volume_exemplo = calcular_volume_tubulacao(0.0603, 0.0525, 120)
    print(f"Exemplo: Volume para 120m de tubo 2\" SCH 40: {volume_exemplo:.4f} m³")
    
    # Executar interface interativa
    main()



In [ ]:
import math

def calcular_volume_tubo(diametro_externo, comprimento, unidades='mm'):
    """
    Calcula o volume de uma tubulação considerando apenas o diâmetro externo.
    Assume que a tubulação é um cilindro sólido (para cálculo de espaço ocupado).
    
    Parâmetros:
    diametro_externo (float): Diâmetro externo da tubulação
    comprimento (float): Comprimento da tubulação
    unidades (str): 'mm' para milímetros ou 'm' para metros
    
    Retorna:
    dict: Dicionário com resultados em diferentes unidades
    """
    try:
        # Converter para metros se necessário
        if unidades == 'mm':
            diametro_externo_m = diametro_externo / 1000
        else:
            diametro_externo_m = diametro_externo
        
        # Calcular raio
        raio = diametro_externo_m / 2
        
        # Calcular volume (V = π × r² × h)
        volume_m3 = math.pi * (raio ** 2) * comprimento
        
        # Converter para outras unidades
        volume_litros = volume_m3 * 1000
        volume_cm3 = volume_m3 * 1000000
        
        return {
            'volume_m3': volume_m3,
            'volume_litros': volume_litros,
            'volume_cm3': volume_cm3,
            'diametro_externo_m': diametro_externo_m,
            'comprimento_m': comprimento
        }
        
    except Exception as e:
        print(f"Erro no cálculo: {e}")
        return None

def calcular_volume_multiplos_tubos(lista_tubos):
    """
    Calcula volume total para múltiplas tubulações
    
    Parâmetros:
    lista_tubos: Lista de dicionários [{'de': valor, 'comprimento': valor, 'unidades': 'mm'/'m'}]
    """
    resultados = []
    volume_total_m3 = 0
    
    for tubo in lista_tubos:
        de = tubo['de']
        comprimento = tubo['comprimento']
        unidades = tubo.get('unidades', 'mm')
        
        resultado = calcular_volume_tubo(de, comprimento, unidades)
        
        if resultado:
            resultados.append({
                'diametro_externo': de,
                'comprimento': comprimento,
                'volume_m3': resultado['volume_m3'],
                'volume_litros': resultado['volume_litros']
            })
            volume_total_m3 += resultado['volume_m3']
    
    return resultados, volume_total_m3

def main():
    print("CALCULADORA DE VOLUME DE TUBULAÇÃO")
    print("=" * 40)
    print("Considera apenas diâmetro externo (cilindro sólido)")
    print("=" * 40)
    
    while True:
        print("\nOpções:")
        print("1 - Calcular volume de uma tubulação")
        print("2 - Calcular volume de múltiplas tubulações")
        print("3 - Tabela de diâmetros padrão")
        print("4 - Sair")
        
        opcao = input("Escolha uma opção (1-4): ").strip()
        
        if opcao == '1':
            try:
                de = ler_float("Diâmetro externo (mm): ")
                comprimento = ler_float("Comprimento (m): ")
                unidades = input("Unidades do diâmetro (mm/m) [padrão: mm]: ").strip().lower() or 'mm'
                
                resultado = calcular_volume_tubo(de, comprimento, unidades)
                
                if resultado:
                    print(f"\n{'='*30}")
                    print("RESULTADOS:")
                    print(f"{'='*30}")
                    print(f"Diâmetro externo: {resultado['diametro_externo_m']:.4f} m")
                    print(f"Comprimento: {comprimento} m")
                    print(f"Volume: {resultado['volume_m3']:.6f} m³")
                    print(f"Volume: {resultado['volume_litros']:.2f} litros")
                    print(f"Volume: {resultado['volume_cm3']:.0f} cm³")
                    print(f"{'='*30}")
                    
            except ValueError:
                print("Erro: Insira valores numéricos válidos.")
                
        elif opcao == '2':
            try:
                num_tubos = int(input("Quantas tubulações deseja calcular? "))
                tubos = []
                
                for i in range(num_tubos):
                    print(f"\nTubulação {i+1}:")
                    de = ler_float("Diâmetro externo (mm): ")
                    comprimento = ler_float("Comprimento (m): ")
                    tubos.append({'de': de, 'comprimento': comprimento, 'unidades': 'mm'})
                
                resultados, volume_total = calcular_volume_multiplos_tubos(tubos)
                
                print(f"\n{'='*50}")
                print("RESULTADOS INDIVIDUAIS:")
                print(f"{'='*50}")
                for i, resultado in enumerate(resultados, 1):
                    print(f"Tubo {i}: DE {resultado['diametro_externo']}mm × {resultado['comprimento']}m")
                    print(f"  Volume: {resultado['volume_m3']:.4f} m³ ({resultado['volume_litros']:.1f} L)")
                
                print(f"\n{'='*30}")
                print(f"VOLUME TOTAL: {volume_total:.4f} m³")
                print(f"VOLUME TOTAL: {volume_total * 1000:.1f} litros")
                print(f"{'='*30}")
                
            except ValueError:
                print("Erro: Insira valores válidos.")
                
        elif opcao == '3':
            # Tabela de diâmetros padrão
            diametros_padrao = {
                '1/2"': 21.3,
                '3/4"': 26.7,
                '1"': 33.4,
                '1½"': 48.3,
                '2"': 60.3,
                '3"': 88.9,
                '4"': 114.3,
                '6"': 168.3,
                '8"': 219.1,
                '10"': 273.0,
                '12"': 323.8
            }
            
            print(f"\n{'='*40}")
            print("DIÂMETROS EXTERNOS PADRÃO (mm)")
            print(f"{'='*40}")
            for tamanho, diametro in diametros_padrao.items():
                print(f"{tamanho:>5} → {diametro} mm")
            print(f"{'='*40}")
            
        elif opcao == '4':
            print("Saindo...")
            break
            
        else:
            print("Opção inválida. Tente novamente.")

# Exemplos de uso direto
if __name__ == "__main__":
    # Exemplo 1: Tubo de 2" (60.3mm) × 100m
    resultado = calcular_volume_tubo(60.3, 100)
    print(f"Exemplo 1 - Tubo 2\" × 100m: {resultado['volume_m3']:.3f} m³")
    
    # Exemplo 2: Múltiplos tubos
    tubos_exemplo = [
        {'de': 60.3, 'comprimento': 50, 'unidades': 'mm'},   # 2" × 50m
        {'de': 114.3, 'comprimento': 30, 'unidades': 'mm'},  # 4" × 30m
        {'de': 219.1, 'comprimento': 20, 'unidades': 'mm'}   # 8" × 20m
    ]
    
    resultados, total = calcular_volume_multiplos_tubos(tubos_exemplo)
    print(f"Exemplo 2 - Volume total: {total:.3f} m³")
    
    # Executar interface interativa
    main()

 